In [1]:
import pandas as pd
from pathlib import Path
import time
import random
import requests
from urllib.parse import quote

In [2]:
output_folder = Path("../data/processed")

In [3]:


animals = pd.DataFrame({
    "animal": [
        "Giant panda",
        "Red panda",
        "Blue whale",
        "Bald eagle",
        "Komodo dragon"
    ],
    "page_title": [
        "Giant_panda",
        "Red_panda",
        "Blue_whale",
        "Bald_eagle",
        "Komodo_dragon"
    ],
    "animal_class": [
        "Mammal",
        "Mammal",
        "Mammal",
        "Bird",
        "Reptile"
    ]
})

data_folder = Path("../data/raw")
data_folder.mkdir(parents=True, exist_ok=True)

animals.to_csv(data_folder / "animals.csv", index = False)

animals

,animal,page_title,animal_class
0,Giant panda,Giant_panda,Mammal
1,Red panda,Red_panda,Mammal
2,Blue whale,Blue_whale,Mammal
3,Bald eagle,Bald_eagle,Bird
4,Komodo dragon,Komodo_dragon,Reptile


In [4]:
# Getting the pageviews
DATA_DIR = Path("../data")
animals = pd.read_csv(DATA_DIR / "raw" / "animals.csv")

start_date = "20250101"
end_date = "20251231"

def get_pageviews(page_title, start, end):
    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/"
        f"per-article/en.wikipedia.org/all-access/user/"
        f"{page_title}/daily/{start}/{end}"
    )

    response = requests.get(
        url,
        headers={"User-Agent": "wikipedia-animals-data-project/1.0"}
    )
    response.raise_for_status()

    records = response.json()["items"]

    return pd.DataFrame({
        "page_title": page_title,
        "date": [record["timestamp"][:8] for record in records],
        "pageviews": [record["views"] for record in records]
    })

In [5]:
animals["page_title"].iloc[2]
get_pageviews(animals["page_title"].iloc[2], start_date, str(int(start_date) + 5))

,page_title,date,pageviews
0,Blue_whale,20250101,2440
1,Blue_whale,20250102,2674
2,Blue_whale,20250103,2619
3,Blue_whale,20250104,3248
4,Blue_whale,20250105,3042
5,Blue_whale,20250106,2560


In [6]:
# All the pageviews across timefor the animals
results = []

for page_title in animals["page_title"]:
    animal_data = get_pageviews(
        page_title,
        start_date,
        end_date
    )

    results.append(animal_data)

pageviews_table = pd.concat(results, ignore_index=True)

pageviews_table

,page_title,date,pageviews
0,Giant_panda,20250101,2808
1,Giant_panda,20250102,2958
2,Giant_panda,20250103,2779
3,Giant_panda,20250104,2849
4,Giant_panda,20250105,2742
...,...,...,...
1820,Komodo_dragon,20251227,2767
1821,Komodo_dragon,20251228,3046
1822,Komodo_dragon,20251229,2536
1823,Komodo_dragon,20251230,2701


In [7]:
# With animal names and classes
pageviews_table = pageviews_table.merge(
    animals,
    on="page_title",
    how="left"
)
pageviews_table

,page_title,date,pageviews,animal,animal_class
0,Giant_panda,20250101,2808,Giant panda,Mammal
1,Giant_panda,20250102,2958,Giant panda,Mammal
2,Giant_panda,20250103,2779,Giant panda,Mammal
3,Giant_panda,20250104,2849,Giant panda,Mammal
4,Giant_panda,20250105,2742,Giant panda,Mammal
...,...,...,...,...,...
1820,Komodo_dragon,20251227,2767,Komodo dragon,Reptile
1821,Komodo_dragon,20251228,3046,Komodo dragon,Reptile
1822,Komodo_dragon,20251229,2536,Komodo dragon,Reptile
1823,Komodo_dragon,20251230,2701,Komodo dragon,Reptile


In [8]:
# The total pageviews
animal_totals = (
    pageviews_table
    .groupby(["animal", "page_title", "animal_class"], as_index=False)
    ["pageviews"]
    .sum()
    .rename(columns={"pageviews": "total_pageviews"})
    .sort_values("total_pageviews", ascending=False)
)

animal_totals

,animal,page_title,animal_class,total_pageviews
2,Giant panda,Giant_panda,Mammal,1167395
3,Komodo dragon,Komodo_dragon,Reptile,1166332
1,Blue whale,Blue_whale,Mammal,922272
4,Red panda,Red_panda,Mammal,899850
0,Bald eagle,Bald_eagle,Bird,780893


In [9]:
# Totals by class
class_totals = (
    animal_totals
    .groupby(["animal_class"], as_index=False)
    ["total_pageviews"]
    .sum()
    .sort_values("total_pageviews", ascending=False)
)
class_totals

,animal_class,total_pageviews
1,Mammal,2989517
2,Reptile,1166332
0,Bird,780893


In [10]:
animals["page_title"].iloc[2]
get_pageviews(animals["page_title"].iloc[2], start_date, str(int(start_date) + 5))

,page_title,date,pageviews
0,Blue_whale,20250101,2440
1,Blue_whale,20250102,2674
2,Blue_whale,20250103,2619
3,Blue_whale,20250104,3248
4,Blue_whale,20250105,3042
5,Blue_whale,20250106,2560


In [11]:
# All the pageviews across timefor the animals
results = []

for page_title in animals["page_title"]:
    animal_data = get_pageviews(
        page_title,
        start_date,
        end_date
    )

    results.append(animal_data)

pageviews_table = pd.concat(results, ignore_index=True)

pageviews_table

,page_title,date,pageviews
0,Giant_panda,20250101,2808
1,Giant_panda,20250102,2958
2,Giant_panda,20250103,2779
3,Giant_panda,20250104,2849
4,Giant_panda,20250105,2742
...,...,...,...
1820,Komodo_dragon,20251227,2767
1821,Komodo_dragon,20251228,3046
1822,Komodo_dragon,20251229,2536
1823,Komodo_dragon,20251230,2701


In [12]:
# With animal names and classes
pageviews_table = pageviews_table.merge(
    animals,
    on="page_title",
    how="left"
)
pageviews_table

,page_title,date,pageviews,animal,animal_class
0,Giant_panda,20250101,2808,Giant panda,Mammal
1,Giant_panda,20250102,2958,Giant panda,Mammal
2,Giant_panda,20250103,2779,Giant panda,Mammal
3,Giant_panda,20250104,2849,Giant panda,Mammal
4,Giant_panda,20250105,2742,Giant panda,Mammal
...,...,...,...,...,...
1820,Komodo_dragon,20251227,2767,Komodo dragon,Reptile
1821,Komodo_dragon,20251228,3046,Komodo dragon,Reptile
1822,Komodo_dragon,20251229,2536,Komodo dragon,Reptile
1823,Komodo_dragon,20251230,2701,Komodo dragon,Reptile


In [13]:
# The total pageviews
animal_totals = (
    pageviews_table
    .groupby(["animal", "page_title", "animal_class"], as_index=False)
    ["pageviews"]
    .sum()
    .rename(columns={"pageviews": "total_pageviews"})
    .sort_values("total_pageviews", ascending=False)
)

animal_totals

,animal,page_title,animal_class,total_pageviews
2,Giant panda,Giant_panda,Mammal,1167395
3,Komodo dragon,Komodo_dragon,Reptile,1166332
1,Blue whale,Blue_whale,Mammal,922272
4,Red panda,Red_panda,Mammal,899850
0,Bald eagle,Bald_eagle,Bird,780893


In [14]:
# Totals by class
class_totals = (
    animal_totals
    .groupby(["animal_class"], as_index=False)
    ["total_pageviews"]
    .sum()
    .sort_values("total_pageviews", ascending=False)
)
class_totals

,animal_class,total_pageviews
1,Mammal,2989517
2,Reptile,1166332
0,Bird,780893


In [15]:
# Now with the full animal table - mammals first

query = """
SELECT DISTINCT ?item ?page_title ?scientific_name WHERE {
  ?item wdt:P31 wd:Q16521;
        wdt:P105 wd:Q7432;
        wdt:P171* wd:Q7377.

  ?article schema:about ?item;
           schema:isPartOf <https://en.wikipedia.org/>;
           schema:name ?page_title.

  OPTIONAL {
    ?item wdt:P225 ?scientific_name.
  }
}
LIMIT 100
"""

url = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

response = requests.get(
    url,
    params={"query": query, "format": "json"},
    headers=headers
)

response.raise_for_status()
results = response.json()["results"]["bindings"]
mammals = pd.DataFrame([
    {
        "animal": row["page_title"]["value"],
        "page_title": row["page_title"]["value"].replace(" ", "_"),
        "animal_class": "Mammal"
    }
    for row in results
])
mammals

,animal,page_title,animal_class
0,Eastern bent-wing bat,Eastern_bent-wing_bat,Mammal
1,Western bent-winged bat,Western_bent-winged_bat,Mammal
2,Small bent-winged bat,Small_bent-winged_bat,Mammal
3,Miniopterus sororculus,Miniopterus_sororculus,Mammal
4,Miniopterus newtoni,Miniopterus_newtoni,Mammal
...,...,...,...
95,Greater short-nosed fruit bat,Greater_short-nosed_fruit_bat,Mammal
96,Micronomus,Micronomus,Mammal
97,Petra fruit bat,Petra_fruit_bat,Mammal
98,Vanikoro flying fox,Vanikoro_flying_fox,Mammal


In [16]:
# Reptiles

query = """
SELECT DISTINCT ?item ?page_title ?scientific_name WHERE {
  ?item wdt:P31 wd:Q16521;
        wdt:P105 wd:Q7432;
        wdt:P171* wd:Q10811.

  ?article schema:about ?item;
           schema:isPartOf <https://en.wikipedia.org/>;
           schema:name ?page_title.

  OPTIONAL {
    ?item wdt:P225 ?scientific_name.
  }
}
LIMIT 100
"""

url = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

response = requests.get(
    url,
    params={"query": query, "format": "json"},
    headers=headers
)

response.raise_for_status()
results = response.json()["results"]["bindings"]
reptiles = pd.DataFrame([
    {
        "animal": row["page_title"]["value"],
        "page_title": row["page_title"]["value"].replace(" ", "_"),
        "animal_class": "Reptile"
    }
    for row in results
])
reptiles

,animal,page_title,animal_class
0,Guarocuyus jaraguanus,Guarocuyus_jaraguanus,Reptile
1,Anomochilus weberi,Anomochilus_weberi,Reptile
2,Anomochilus leonardi,Anomochilus_leonardi,Reptile
3,Indian star tortoise,Indian_star_tortoise,Reptile
4,Burmese star tortoise,Burmese_star_tortoise,Reptile
...,...,...,...
95,Texas tortoise,Texas_tortoise,Reptile
96,Anomochilus monticola,Anomochilus_monticola,Reptile
97,Common box turtle,Common_box_turtle,Reptile
98,Forsten's tortoise,Forsten's_tortoise,Reptile


In [17]:
# Fish - bony and cartiliginous classes combined here

query = """
SELECT DISTINCT ?item ?page_title ?scientific_name WHERE {
  VALUES ?fish_group {
    wd:Q127282
    wd:Q25371
  }

  ?item wdt:P31 wd:Q16521;
        wdt:P105 wd:Q7432;
        wdt:P171* ?fish_group.

  ?article schema:about ?item;
           schema:isPartOf <https://en.wikipedia.org/>;
           schema:name ?page_title.

  OPTIONAL {
    ?item wdt:P225 ?scientific_name.
  }
}
LIMIT 100
"""

url = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

response = requests.get(
    url,
    params={"query": query, "format": "json"},
    headers=headers
)

response.raise_for_status()
results = response.json()["results"]["bindings"]
fish = pd.DataFrame([
    {
        "animal": row["page_title"]["value"],
        "page_title": row["page_title"]["value"].replace(" ", "_"),
        "animal_class": "Fish"
    }
    for row in results
])
fish

,animal,page_title,animal_class
0,Betta edithae,Betta_edithae,Fish
1,Betta brownorum,Betta_brownorum,Fish
2,Betta channoides,Betta_channoides,Fish
3,Peaceful betta,Peaceful_betta,Fish
4,John Dory,John_Dory,Fish
...,...,...,...
95,Barbeled houndshark,Barbeled_houndshark,Fish
96,European plaice,European_plaice,Fish
97,Beaufortia kweichowensis,Beaufortia_kweichowensis,Fish
98,Blue shark,Blue_shark,Fish


In [18]:
# Birds

query = """
SELECT DISTINCT ?item ?page_title ?scientific_name WHERE {
  ?item wdt:P31 wd:Q16521;
        wdt:P105 wd:Q7432;
        wdt:P171+ wd:Q5113.

  ?article schema:about ?item;
           schema:isPartOf <https://en.wikipedia.org/>;
           schema:name ?page_title.

  OPTIONAL {
    ?item wdt:P225 ?scientific_name.
  }
}
LIMIT 100
"""

url = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

response = requests.get(
    url,
    params={"query": query, "format": "json"},
    headers=headers
)

response.raise_for_status()
results = response.json()["results"]["bindings"]
birds = pd.DataFrame([
    {
        "animal": row["page_title"]["value"],
        "page_title": row["page_title"]["value"].replace(" ", "_"),
        "animal_class": "Bird"
    }
    for row in results
])
birds

,animal,page_title,animal_class
0,Wild turkey,Wild_turkey,Bird
1,Ocellated turkey,Ocellated_turkey,Bird
2,Green-crowned plovercrest,Green-crowned_plovercrest,Bird
3,Banded quail,Banded_quail,Bird
4,Southern giant hummingbird,Southern_giant_hummingbird,Bird
...,...,...,...
95,Blue-throated mountaingem,Blue-throated_mountaingem,Bird
96,Razor-billed curassow,Razor-billed_curassow,Bird
97,Green-backed firecrown,Green-backed_firecrown,Bird
98,Emerald-chinned hummingbird,Emerald-chinned_hummingbird,Bird


In [19]:
# Amphibians

query = """
SELECT DISTINCT ?item ?page_title ?scientific_name WHERE {
  ?item wdt:P31 wd:Q16521;
        wdt:P105 wd:Q7432;
        wdt:P171+ wd:Q10908.

  ?article schema:about ?item;
           schema:isPartOf <https://en.wikipedia.org/>;
           schema:name ?page_title.

  OPTIONAL {
    ?item wdt:P225 ?scientific_name.
  }
}
LIMIT 100
"""

url = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

response = requests.get(
    url,
    params={"query": query, "format": "json"},
    headers=headers
)

response.raise_for_status()
results = response.json()["results"]["bindings"]
amphibians = pd.DataFrame([
    {
        "animal": row["page_title"]["value"],
        "page_title": row["page_title"]["value"].replace(" ", "_"),
        "animal_class": "Amphibian"
    }
    for row in results
])
amphibians

,animal,page_title,animal_class
0,Marmorerpeton wakei,Marmorerpeton_wakei,Amphibian
1,Boulengerula taitana,Boulengerula_taitana,Amphibian
2,Caecilia volcani,Caecilia_volcani,Amphibian
3,Siphonops annulatus,Siphonops_annulatus,Amphibian
4,Stefania roraimae,Stefania_roraimae,Amphibian
...,...,...,...
95,California giant salamander,California_giant_salamander,Amphibian
96,Coastal giant salamander,Coastal_giant_salamander,Amphibian
97,Red-crowned toadlet,Red-crowned_toadlet,Amphibian
98,Ptychadena boettgeri,Ptychadena_boettgeri,Amphibian


In [20]:
# New animals table
animals = pd.concat(
    [mammals, reptiles, birds, fish, amphibians], 
    ignore_index = True)
animals.to_csv(
    output_folder / "animals",
    index = False
)
animals

,animal,page_title,animal_class
0,Eastern bent-wing bat,Eastern_bent-wing_bat,Mammal
1,Western bent-winged bat,Western_bent-winged_bat,Mammal
2,Small bent-winged bat,Small_bent-winged_bat,Mammal
3,Miniopterus sororculus,Miniopterus_sororculus,Mammal
4,Miniopterus newtoni,Miniopterus_newtoni,Mammal
...,...,...,...
495,California giant salamander,California_giant_salamander,Amphibian
496,Coastal giant salamander,Coastal_giant_salamander,Amphibian
497,Red-crowned toadlet,Red-crowned_toadlet,Amphibian
498,Ptychadena boettgeri,Ptychadena_boettgeri,Amphibian


In [21]:
# Image count as a third variable

session = requests.Session()

image_headers = {
    "User-Agent": (
        "wikipedia-animals/1.0 "
        "(educational project; contact: noabm18@gmail.com)"
    )
}

def get_image_count(page_title, max_retries=8):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "formatversion": 2,
        "prop": "images",
        "titles": page_title.replace("_", " "),
        "imlimit": "max"
    }

    image_count = 0

    while True:
        for attempt in range(max_retries):
            response = session.get(
                url,
                params=params,
                headers=image_headers,
                timeout=30
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after and retry_after.isdigit():
                    wait_time = int(retry_after)
                else:
                    wait_time = min(2 ** attempt, 120)

                wait_time += random.uniform(0, 1)

                print(
                    f"Rate limited on {page_title}. "
                    f"Waiting {wait_time:.1f} seconds."
                )

                time.sleep(wait_time)
                continue

            response.raise_for_status()
            break

        else:
            raise RuntimeError(
                f"Failed to retrieve {page_title} "
                f"after {max_retries} attempts"
            )

        data = response.json()
        page = data["query"]["pages"][0]
        
        image_count += len(page.get("images", []))

        if "continue" not in data:
            return image_count

        params.update(data["continue"])

        time.sleep(2)

In [22]:
get_image_count("bald_eagle")

43

In [23]:
image_results = []

for number, page_title in enumerate(
    animals["page_title"],
    start=1
):
    try:
        image_count = get_image_count(page_title)

        image_results.append({
            "page_title": page_title,
            "image_count": image_count
        })

        print(
            f"{number}/{len(animals)}: "
            f"{page_title} has {image_count} images"
        )

    except requests.RequestException as error:
        print(f"Skipped {page_title}: {error}")

    time.sleep(1)

1/500: Eastern_bent-wing_bat has 3 images
2/500: Western_bent-winged_bat has 4 images
3/500: Small_bent-winged_bat has 5 images
4/500: Miniopterus_sororculus has 3 images
5/500: Miniopterus_newtoni has 3 images
6/500: Manavi_long-fingered_bat has 5 images
7/500: Peterson's_long-fingered_bat has 5 images
8/500: Miniopterus_aelleni has 4 images
9/500: Common_bent-wing_bat has 6 images
10/500: Andean_caenolestid has 6 images
11/500: Gray-bellied_caenolestid has 8 images
12/500: Madagascan_flying_fox has 4 images
13/500: Seychelles_fruit_bat has 5 images
14/500: Aru_flying_fox has 4 images
15/500: Dusky_caenolestid has 9 images
16/500: Northern_caenolestid has 8 images
17/500: Long-haired_fruit_bat has 6 images
18/500: Franquet's_epauletted_fruit_bat has 5 images
19/500: Fijian_monkey-faced_bat has 4 images
20/500: Madagascar_sucker-footed_bat has 4 images
21/500: Madagascar_free-tailed_bat has 3 images
22/500: Greater_monkey-faced_bat has 3 images
23/500: African_sheath-tailed_bat has 5 i

In [24]:
image_table = pd.DataFrame(image_results)

image_table

,page_title,image_count
0,Eastern_bent-wing_bat,3
1,Western_bent-winged_bat,4
2,Small_bent-winged_bat,5
3,Miniopterus_sororculus,3
4,Miniopterus_newtoni,3
...,...,...
495,California_giant_salamander,8
496,Coastal_giant_salamander,4
497,Red-crowned_toadlet,5
498,Ptychadena_boettgeri,3


In [25]:
# A new get_pageviews because it easily gets timeout errors for this many queries

session = requests.Session()

pageview_headers = {
    "User-Agent": "wikipedia-animals/1.0 (educational data science project)"
}

def get_pageviews(page_title, start, end, max_retries=6):
    encoded_title = quote(
        page_title.replace(" ", "_"),
        safe=""
    )

    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/"
        f"per-article/en.wikipedia.org/all-access/user/"
        f"{encoded_title}/daily/{start}/{end}"
    )

    for attempt in range(max_retries):
        response = session.get(
            url,
            headers=pageview_headers,
            timeout=30
        )

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After")

            if retry_after and retry_after.isdigit():
                wait_time = int(retry_after)
            else:
                wait_time = 2 ** attempt

            print(
                f"Rate limited on {page_title}. "
                f"Waiting {wait_time} seconds."
            )

            time.sleep(wait_time)
            continue

        response.raise_for_status()

        records = response.json()["items"]

        return pd.DataFrame({
            "page_title": page_title,
            "date": [
                record["timestamp"][:8]
                for record in records
            ],
            "pageviews": [
                record["views"]
                for record in records
            ]
        })

    raise RuntimeError(
        f"Could not retrieve {page_title} after {max_retries} attempts"
    )

In [26]:
# All the pageviews across timefor the animals - now with the full table

pageview_results = []

for number, page_title in enumerate(
    animals["page_title"],
    start=1
):
    try:
        animal_data = get_pageviews(
            page_title,
            start_date,
            end_date
        )

        pageview_results.append(animal_data)

        print(
            f"{number}/{len(animals)}: "
            f"downloaded {page_title}"
        )

    except requests.RequestException as error:
        print(f"Skipped {page_title}: {error}")

    except RuntimeError as error:
        print(error)

    time.sleep(1)

1/500: downloaded Eastern_bent-wing_bat
2/500: downloaded Western_bent-winged_bat
3/500: downloaded Small_bent-winged_bat
4/500: downloaded Miniopterus_sororculus
5/500: downloaded Miniopterus_newtoni
6/500: downloaded Manavi_long-fingered_bat
7/500: downloaded Peterson's_long-fingered_bat
8/500: downloaded Miniopterus_aelleni
9/500: downloaded Common_bent-wing_bat
10/500: downloaded Andean_caenolestid
Rate limited on Gray-bellied_caenolestid. Waiting 23 seconds.
11/500: downloaded Gray-bellied_caenolestid
12/500: downloaded Madagascan_flying_fox
13/500: downloaded Seychelles_fruit_bat
14/500: downloaded Aru_flying_fox
15/500: downloaded Dusky_caenolestid
16/500: downloaded Northern_caenolestid
17/500: downloaded Long-haired_fruit_bat
18/500: downloaded Franquet's_epauletted_fruit_bat
19/500: downloaded Fijian_monkey-faced_bat
20/500: downloaded Madagascar_sucker-footed_bat
Rate limited on Madagascar_free-tailed_bat. Waiting 49 seconds.
21/500: downloaded Madagascar_free-tailed_bat
22/

In [27]:

pageviews_table = pd.concat(
    pageview_results,
    ignore_index=True
)


In [28]:
# With animal names and classes
animals_full = pageviews_table.merge(
    animals,
    on="page_title",
    how="left"
).merge(
    image_table,
    on="page_title",
    how="left"
)
animals_full.to_csv(
    output_folder / "animals_full.csv",
    index=False
)
animals_full

,page_title,date,pageviews,animal,animal_class,image_count
0,Eastern_bent-wing_bat,20250101,2,Eastern bent-wing bat,Mammal,3
1,Eastern_bent-wing_bat,20250102,0,Eastern bent-wing bat,Mammal,3
2,Eastern_bent-wing_bat,20250103,0,Eastern bent-wing bat,Mammal,3
3,Eastern_bent-wing_bat,20250104,0,Eastern bent-wing bat,Mammal,3
4,Eastern_bent-wing_bat,20250105,3,Eastern bent-wing bat,Mammal,3
...,...,...,...,...,...,...
166509,Leptobrachella_aerea,20251227,2,Leptobrachella aerea,Amphibian,2
166510,Leptobrachella_aerea,20251228,1,Leptobrachella aerea,Amphibian,2
166511,Leptobrachella_aerea,20251229,2,Leptobrachella aerea,Amphibian,2
166512,Leptobrachella_aerea,20251230,2,Leptobrachella aerea,Amphibian,2
